# Observed Potomac daily flow, 2021–2024

This notebook checks the pinned USGS daily-mean response and the two periods declared before retrieval. It uses only Python's standard library and Earth Rehearsal's original source. It is descriptive observational analysis, not climate attribution, flood prediction, field validation or an intervention-effect study.

Source: U.S. Geological Survey, USGS Water Data for the Nation, accessed September 8, 2026, database DOI [10.5066/F7P55KJN](https://doi.org/10.5066/F7P55KJN). Station USGS-01646500. Raw API data is US Government work in the public domain; original notebook/code is AGPL-3.0-only. Exact requests and terms are retained in the source record.


In [1]:
import json, hashlib, sys
from pathlib import Path
from collections import Counter
from datetime import date, timedelta
from decimal import Decimal, localcontext
root = Path.cwd()
if not (root / 'earth.py').is_file():
    root = root.parent
assert (root / 'earth.py').is_file(), 'Run from the project or its notebooks directory'
sys.path.insert(0, str(root / 'src'))
snapshot = root / 'scenarios/observations/potomac-2021-2024'
source = json.loads((snapshot / 'source.json').read_text())
study = json.loads((snapshot / 'study.json').read_text())
raw_bytes = (snapshot / 'daily.json').read_bytes()
assert hashlib.sha256(raw_bytes).hexdigest() == source['retrievals']['daily']['sha256']
raw = json.loads(raw_bytes)
print('Pinned bytes:', len(raw_bytes))
print('Source SHA-256:', hashlib.sha256(raw_bytes).hexdigest())
print('Retrieval:', source['retrievals']['daily']['retrieved_utc'])
print('Date scope:', study['start_date'], 'through', study['end_date'])


Pinned bytes: 980465
Source SHA-256: 0057a418728591af3d2915f5c10df198f3e4e18189e97d726c26dc8594ee758d
Retrieval: 2026-09-08T16:18:14.089029+00:00
Date scope: 2021-01-01 through 2024-12-31


## Grain, quality and missingness

The grain is one daily mean per civil observation date, one time series, one parameter and one point station. A date is not converted into an invented UTC instant. An absent day and a source row with a null value are different. Qualifier counts may overlap.


In [2]:
records = [feature['properties'] for feature in raw['features']]
dates = [record['time'] for record in records]
expected_days = (date.fromisoformat(source['end_date']) - date.fromisoformat(source['start_date'])).days + 1
assert len(set(dates)) == len(dates), 'Duplicate daily grain'
assert len({record['time_series_id'] for record in records}) == 1
assert {record['parameter_code'] for record in records} == {'00060'}
assert {record['statistic_id'] for record in records} == {'00003'}
assert {record['unit_of_measure'] for record in records} == {'ft^3/s'}
print('Records / requested days:', len(records), '/', expected_days)
print('Date range:', min(dates), max(dates))
print('Absent days:', expected_days - len(records))
print('Null-valued records:', sum(record['value'] is None for record in records))
print('Approval counts:', dict(Counter(record['approval_status'] for record in records)))
print('Qualifier combinations:', dict(Counter(','.join(record['qualifier'] or []) or 'None' for record in records)))


Records / requested days: 1461 / 1461
Date range: 2021-01-01 2024-12-31
Absent days: 0
Null-valued records: 0
Approval counts: {'Approved': 1461}
Qualifier combinations: {'None': 1319, 'REVISED': 112, 'ESTIMATED': 26, 'ESTIMATED,REVISED': 4}


## Separate arithmetic and quality sensitivity

Primary summaries include nonnegative approved observations except estimates. The sensitivity additionally includes approved estimates. Provisional and negative values are retained as excluded. No interpolation, filling, outlier removal or significance test is used. The conversion below is independently expressed as the cube of the exact international-foot length (0.3048 m); it does not add measurement precision.


In [3]:
with localcontext() as context:
    context.prec = 80
    factor = Decimal('0.3048') ** 3
    period_means = []
    for period in study['periods']:
        primary, with_estimates = [], []
        for record in records:
            if not period['start_date'] <= record['time'] <= period['end_date']:
                continue
            if record['value'] is None or record['approval_status'] != 'Approved':
                continue
            value = Decimal(record['value'])
            if value < 0:
                continue
            with_estimates.append(value * factor)
            if 'ESTIMATED' not in (record['qualifier'] or []):
                primary.append(value * factor)
        mean = sum(primary) / len(primary) if primary else None
        sensitivity_mean = sum(with_estimates) / len(with_estimates) if with_estimates else None
        print(period['id'], '| primary days:', len(primary), '| mean m3/s:', round(mean, 6) if mean is not None else None,
              '| with estimates days:', len(with_estimates), '| mean:', round(sensitivity_mean, 6) if sensitivity_mean is not None else None)
        period_means.append(mean)
    print('Second minus first period mean, m3/s:', round(period_means[1] - period_means[0], 6))


2021-2022 | primary days: 712 | mean m3/s: 260.789010 | with estimates days: 730 | mean: 259.109620
2023-2024 | primary days: 719 | mean m3/s: 227.718291 | with estimates days: 731 | mean: 225.841029
Second minus first period mean, m3/s: -33.070719


In [4]:
from earth_rehearsal.observed_data import normalize, validate_study, analyze
from earth_rehearsal.observed_reference import verify_statistics
site_bytes = (snapshot / 'site.json').read_bytes()
station, days = normalize(source, raw_bytes, site_bytes)
validate_study(study, source)
summary = analyze(study, station, days)
reference = verify_statistics(study, raw_bytes, summary)
print('Separate evaluator:', reference['groups_checked'], 'groups and', reference['policies_checked'], 'quality policies checked')
for group in summary['summaries']:
    if group['kind'] == 'period':
        print(group['id'], '| mean:', round(group['primary']['mean_m3_s'], 3), '| median:', round(group['primary']['median_m3_s'], 3),
              '| p90:', round(group['primary']['p90_m3_s'], 3), '| coverage:', round(100 * group['primary']['coverage_fraction'], 2), '%')
assert all(abs(float(expected) - group['primary']['mean_m3_s']) < 1e-9 for expected, group in zip(period_means, [g for g in summary['summaries'] if g['kind'] == 'period']))


Separate evaluator: 55 groups and 2 quality policies checked
2021-2022 | mean: 260.789 | median: 171.742 | p90: 475.157 | coverage: 97.53 %
2023-2024 | mean: 227.718 | median: 133.656 | p90: 510.27 | coverage: 98.36 %


## Interpretation and reproducible outputs

All 1,461 requested dates are present; 30 approved estimates are excluded from the primary record and retained in the sensitivity. The finite-window mean is lower in 2023–2024 than in 2021–2022, while the empirical p90 is higher. Different summaries describe different parts of the distribution. These four selected years and this one station do not establish a regional or climate trend.

Use `python3 earth.py observations run --out runs/potomac` to create the complete offline report, raw-data bundle, normalized CSV and summary tables. Inspect with `python3 earth.py observations inspect runs/potomac`. Reproduction requires the matching source runtime and a fresh output directory. The original source files, qualifiers and dates stay in the bundle. Review requirements and the full 221-task programme remain separate from these automated checks.
